# Multilingual Health QA — V7 Reproducible Solution (0.6908)

**Author:** Samuel Mwania (mwaniasam)  
**Competition:** Multilingual Health Question Answering in Low-Resource African Languages by ITU  
**Final Score:** 0.6908 (public LB) — Rank 15  

## Solution Architecture

```
AfriE5-Large-instruct (fine-tuned) → Top-15 candidates → Per-language routing:
  ├── Eng_Uga, Eng_Ken, Eng_Eth → Embedding interpolation → MBR top-1
  ├── Swa_Ken → AfriE5 + FT2 interpolation (β=0.8) → MBR top-1
  ├── Lug_Uga → AfriE5 + per-lang adapter (β=0.8) → MBR top-1
  ├── Aka_Gha, Amh_Eth → CE reranker → extractive stitch
  └── Eng_Gha → Qwen2.5-7B LoRA generation
```

All components are fully open-source. No paid APIs.

## How to run

1. Mount Google Drive (contains pre-trained models + cached embeddings)
2. Run all cells in order
3. Output: `submission_v7.csv` (2618 rows, identical answer columns)

**GPU required:** T4 or better (CE reranker + Qwen inference)  
**Runtime:** ~20 minutes (inference only, all models pre-trained)


## 1. Install Dependencies


In [ ]:
!pip install -q -U torchao pylcs faiss-cpu rouge-score sentence-transformers transformers accelerate bitsandbytes peft


## 2. Bootstrap: Load Data, Embeddings, and Pipeline State

This cell restores the complete pipeline state from Google Drive:
- **Data:** Train (29,815), Val (6,686), Test (2,618) question-answer pairs
- **Embeddings:** AfriE5, Gemini-embed, BGE-M3, answer embeddings (all cached as .npy)
- **FAISS indices:** Per-language inner-product indices for fast retrieval
- **Pickled state:** Pre-computed candidate pools, validation scores, tuned parameters
- **Core functions:** Unicode ROUGE scorer, MBR selection, extractive stitcher

All helper functions are defined here. The tuned per-language decisions (choice table, 
stitch gates) are hardcoded for exact reproducibility.


In [ ]:
# =============================================================================
# BOOTSTRAP: restore full pipeline state from Drive caches (~5-10 min, CPU ok)
# Run this first after ANY runtime restart.
# =============================================================================
import os, re as _re, json, pickle, gc
import numpy as np, pandas as pd, faiss
from pathlib import Path
from tqdm import tqdm
from collections import Counter
from rouge_score import rouge_scorer
from rouge_score import tokenize as rs_tokenize
try:
    import pylcs; HAVE_PYLCS = True
except ImportError:
    os.system('pip install -q pylcs')
    try: import pylcs; HAVE_PYLCS = True
    except Exception: HAVE_PYLCS = False

from google.colab import drive
if not Path('/content/drive/MyDrive').exists(): drive.mount('/content/drive')
BASE = Path('/content/drive/MyDrive/multilingual-health-qa')
DATA_DIR, OUTPUT_DIR = BASE/'data', BASE/'outputs'
CACHE = OUTPUT_DIR/'mbr_cache'

# ---- data ----
train_df = pd.read_csv(DATA_DIR/'Train.csv')
val_df   = pd.read_csv(DATA_DIR/'Val.csv')
test_df  = pd.read_csv(DATA_DIR/'Test.csv')
sample_sub = pd.read_csv(DATA_DIR/'SampleSubmission.csv'); SUB_COLS = list(sample_sub.columns)
FT_MODEL_DIR = OUTPUT_DIR / 'qwen-ft-health'
combined = pd.concat([train_df, val_df], ignore_index=True).dropna(subset=['input','output']).reset_index(drop=True)
questions_raw = combined['input'].astype(str).tolist()
answers_raw   = combined['output'].astype(str).tolist()
subsets_raw   = combined['subset'].astype(str).tolist()
corpus_q_stripped = [q.strip() for q in questions_raw]
val_qs  = val_df['input'].fillna('').astype(str).tolist()
test_qs = test_df['input'].fillna('').astype(str).tolist()
test_subs = test_df['subset'].fillna('').astype(str).tolist()
SUBSET_TO_LANG = {'Aka_Gha':'Akan (Ghana)','Amh_Eth':'Amharic (Ethiopia)','Eng_Eth':'English (Ethiopia)',
 'Eng_Gha':'English (Ghana)','Eng_Ken':'English (Kenya)','Eng_Uga':'English (Uganda)',
 'Lug_Uga':'Luganda (Uganda)','Swa_Ken':'Swahili (Kenya)'}
scorer_both = rouge_scorer.RougeScorer(['rouge1','rougeL'], use_stemmer=False)

# ---- embeddings (all cached) ----
corpus_emb = np.load(CACHE/'emb_corpus.npy'); val_emb = np.load(CACHE/'emb_val.npy'); test_emb = np.load(CACHE/'emb_test.npy')
gem_corpus = np.load(CACHE/'gem_emb_corpus.npy'); gem_val = np.load(CACHE/'gem_emb_val.npy'); gem_test = np.load(CACHE/'gem_emb_test.npy')
ans_emb = np.load(CACHE/'emb_corpus_answers.npy')
bge_corpus = np.load(CACHE/'bge_corpus.npy'); bge_val = np.load(CACHE/'bge_val.npy'); bge_test = np.load(CACHE/'bge_test.npy')

# ---- indices ----
def build_lang_idx(emb):
    out = {}
    for sub in sorted(set(subsets_raw)):
        mask = [i for i,s in enumerate(subsets_raw) if s==sub]
        ix = faiss.IndexFlatIP(emb.shape[1]); ix.add(emb[mask]); out[sub]=(ix,mask)
    return out
lang_indices = build_lang_idx(corpus_emb); gem_lang_idx = build_lang_idx(gem_corpus)
qa_idx = build_lang_idx(ans_emb); bge_idx = build_lang_idx(bge_corpus)
global_idx = faiss.IndexFlatIP(corpus_emb.shape[1]); global_idx.add(corpus_emb)

# ---- pickled state ----
val_cands_all = pickle.load(open(CACHE/'val_cands.pkl','rb'))
val_prep, val_refscores = pickle.load(open(CACHE/'val_prep.pkl','rb'))
v4c, v4p, v4r = pickle.load(open(CACHE/'val_union4.pkl','rb'))
P = pickle.load(open(CACHE/'uni_rebuild.pkl','rb'))
llm_ans = json.load(open(OUTPUT_DIR/'gemini_mbr_llm_prog.json'))

# ---- core functions (canonical definitions) ----
K_CANDIDATES, K_LEG, CAP = 15, 20, 400
_UNI = _re.compile(r'\w+', _re.UNICODE)
def uni_toks(t): return _UNI.findall(t.lower())
def _lcs_py(a,b):
    if not a or not b: return 0
    dp=[0]*(len(b)+1)
    for ai in a:
        prev=0
        for j,bj in enumerate(b):
            cur=dp[j+1]; dp[j+1]=prev+1 if ai==bj else max(dp[j+1],dp[j]); prev=cur
    return dp[-1]
def lcs_tok(a,b):
    if HAVE_PYLCS:
        v={}
        for t in a:
            if t not in v: v[t]=len(v)
        for t in b:
            if t not in v: v[t]=len(v)
        return pylcs.lcs_sequence_length(''.join(chr(0x100+v[t]) for t in a), ''.join(chr(0x100+v[t]) for t in b))
    return _lcs_py(a,b)
def uni_r1(rt,ht):
    if not rt or not ht: return 0.0
    return 2*sum((Counter(rt)&Counter(ht)).values())/(len(rt)+len(ht))
def uni_rl(rt,ht):
    if not rt or not ht: return 0.0
    return 2*lcs_tok(rt,ht)/(len(rt)+len(ht))
def get_same_lang_candidates(q_text,q_emb,subset,k=K_CANDIDATES,exclude_exact=True):
    qs=q_text.strip()
    if subset in lang_indices:
        idx,mask=lang_indices[subset]
        D,I=idx.search(np.asarray(q_emb,np.float32).reshape(1,-1),min(k+5,len(mask)))
        out=[]
        for d,li in zip(D[0],I[0]):
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            out.append({'answer':answers_raw[ci],'sim':float(d),'idx':ci})
            if len(out)>=k: break
        if out: return out
    return []
def union4(q_text,afri_q,gem_q,bge_q,subset,exclude_exact=True):
    qs=q_text.strip(); rrf={}
    for idx_map,emb in [(lang_indices,afri_q),(gem_lang_idx,gem_q),(qa_idx,afri_q),(bge_idx,bge_q)]:
        if subset not in idx_map: continue
        idx,mask=idx_map[subset]
        D,I=idx.search(np.asarray(emb,np.float32).reshape(1,-1),min(K_LEG+5,len(mask)))
        r=0
        for li in I[0]:
            if li<0: continue
            ci=mask[int(li)]
            if exclude_exact and corpus_q_stripped[ci]==qs: continue
            rrf[ci]=rrf.get(ci,0.0)+1.0/(60+r); r+=1
            if r>=K_LEG: break
    ranked=sorted(rrf.items(),key=lambda kv:-kv[1])[:35]
    return [{'answer':answers_raw[ci],'sim':float(np.dot(afri_q,corpus_emb[ci])),'idx':ci} for ci,_ in ranked]
def uni_prep(cands,max_tok=80):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    seen,dd,ddw={},[],[]
    for a,wi in zip(answers,w):
        k=a.strip().lower()
        if k in seen: ddw[seen[k]]+=wi
        else: seen[k]=len(dd); dd.append(a); ddw.append(wi)
    ddw=np.array(ddw); ddw/=ddw.sum(); n=len(dd)
    toks=[uni_toks(a)[:max_tok] for a in dd]
    if n==1: return dd,ddw,np.zeros(1),np.zeros(1)
    S1,SL=np.zeros((n,n)),np.zeros((n,n))
    for i in range(n):
        for j in range(i+1,n):
            S1[i,j]=S1[j,i]=uni_r1(toks[i],toks[j]); SL[i,j]=SL[j,i]=uni_rl(toks[i],toks[j])
    return dd,ddw,S1@ddw,SL@ddw
def mbr_idx(ub,ddw,alpha,margin):
    if len(ddw)==1: return 0
    u=ub+alpha*ddw; b=int(np.argmax(u))
    return b if (b!=0 and u[b]-u[0]>margin) else 0
uni_prior={sub: float(np.median([len(uni_toks(answers_raw[i])) for i,s in enumerate(subsets_raw) if s==sub])) for sub in SUBSET_TO_LANG}
def uni_stitch(cands,lam,sub):
    answers=[c['answer'] for c in cands]
    w=np.exp(np.array([c['sim'] for c in cands])*5); w/=w.sum()
    p=Counter()
    for a,wi in zip(answers,w):
        for t,c in Counter(uni_toks(a)[:CAP]).items(): p[t]+=wi*c
    ref_len=max(lam*uni_prior[sub],1.0); seen,pool=set(),[]
    for a in answers:
        for s in _re.split(r'(?<=[.!?])\s+|\n+',a):
            s=s.strip(); st=uni_toks(s)
            if len(st)<4: continue
            k=' '.join(st)
            if k not in seen: seen.add(k); pool.append((s,Counter(st),len(st)))
    if not pool: return answers[0]
    def ef1(cov,hl):
        mt=sum(min(c,p[t]) for t,c in cov.items())
        if hl==0 or mt==0: return 0.0
        pr_,rc=mt/hl,mt/ref_len; return 2*pr_*rc/(pr_+rc)
    chosen,cov,hl,cur=[],Counter(),0,0.0
    while pool:
        bi,bg=-1,0.0
        for i,(s,sc_,sl) in enumerate(pool):
            nc=cov.copy(); nc.update(sc_)
            g=ef1(nc,hl+sl)-cur
            if g>bg: bg,bi=g,i
        if bi<0: break
        s,sc_,sl=pool.pop(bi); chosen.append(s); cov.update(sc_); hl+=sl; cur+=bg
    return ' '.join(chosen) if chosen else answers[0]

# ---- tuned decisions from the unicode rebuild (hardcoded = reproducible) ----
choice = {'Aka_Gha':('2leg',0.10,0.010),'Amh_Eth':('2leg',0.05,99.0),'Eng_Eth':('2leg',0.05,99.0),
          'Eng_Gha':('4leg',0.20,0.050),'Eng_Ken':('2leg',0.05,99.0),'Eng_Uga':('2leg',0.05,99.0),
          'Lug_Uga':('2leg',0.05,99.0),'Swa_Ken':('2leg',0.05,99.0)}
uni_stitch_gate = {'Aka_Gha':{'use':True,'lam':0.70,'pool':'2leg'},
                   'Eng_Gha':{'use':True,'lam':0.85,'pool':'4leg'},
                   'Amh_Eth':{'use':True,'lam':0.70,'pool':'2leg'}}
print("STATE RESTORED:", len(combined), "corpus |", len(llm_ans), "LLM answers | all caches loaded")

## 3. Load Pre-trained Models

Three models are loaded from Drive:
1. **AfriE5-Large-instruct** (sentence-transformers) — the retrieval backbone
2. **CE Reranker v2** (xlm-roberta-base) — binary classifier for candidate reranking
3. **Qwen2.5-7B + LoRA** — generative reader for Eng_Gha only

If the CE model exists on Drive, we load it directly. Otherwise, we train it from scratch (Cell 4).


In [ ]:
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# 1. AfriE5-Large (retrieval backbone)
enc_model = SentenceTransformer(str(OUTPUT_DIR / 'afrie5-ft-health'))
print(f"AfriE5 loaded: {enc_model.get_sentence_embedding_dimension()}d")

# 2. Cross-encoder reranker v2
CE_DIR = OUTPUT_DIR / 'ce-reranker-v2'
if CE_DIR.exists():
    ctok = AutoTokenizer.from_pretrained(str(CE_DIR))
    cmod = AutoModelForSequenceClassification.from_pretrained(str(CE_DIR)).to('cuda:0').eval()
    print(f"CE reranker loaded from cache")
    CE_NEEDS_TRAINING = False
else:
    print("CE reranker not found — will train in Cell 4")
    CE_NEEDS_TRAINING = True

# CE inference function
@torch.no_grad()
def ce_scores(query, cand_questions):
    enc = ctok([query]*len(cand_questions), cand_questions, padding=True,
               truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmod(**enc).logits, -1)[:, 1].cpu().numpy()


## 4. Cross-Encoder Training (skip if loaded from cache)

Trains `xlm-roberta-base` as a binary classifier:
- **Positive:** candidate with ROUGE-1 ≥ 0.5 against validation reference
- **Negative:** candidate with ROUGE-1 ≤ 0.2
- **Labels computed with unicode tokenizer** (critical fix — broken tokenizer corrupted ~40% of labels)
- **1 epoch, lr=2e-5, batch size 16**


In [ ]:
if CE_NEEDS_TRAINING:
    # =============================================================================
    # CROSS-ENCODER RERANKER v2 — done right: classification, unicode labels, guarded
    # Train: XLM-R-base on (query, candidate-question) pairs. Deploy: rerank top-15,
    # per-language gate + margin override. L4/T4, ~2-3 hrs total.
    # =============================================================================
    !pip install -q -U transformers accelerate
    import torch, gc, random, numpy as np
    from tqdm import tqdm
    random.seed(42); np.random.seed(42)

    # ---- 1) MINE PAIRS (unicode labels, train rows only, all languages) ----
    train_q_set = set(q.strip() for q in train_df['input'].dropna().astype(str))
    is_train_row = np.array([q in train_q_set for q in corpus_q_stripped])
    pairs = []   # (query_text, cand_text, label)
    PER_LANG = 2500
    for sub in SUBSET_TO_LANG:
        idx_t = [i for i in range(len(combined)) if subsets_raw[i]==sub and is_train_row[i]]
        random.shuffle(idx_t)
        ix, mask = lang_indices[sub]; mask_arr = np.array(mask)
        tmask = np.array([is_train_row[ci] for ci in mask])
        made = 0
        for qi in idx_t:
            if made >= PER_LANG: break
            gold = uni_toks(answers_raw[qi])[:CAP]
            if len(gold) < 3: continue
            D, I = ix.search(corpus_emb[qi].reshape(1,-1), 20)
            pos, neg = [], []
            for li in I[0]:
                if li < 0 or not tmask[int(li)]: continue
                ci = int(mask_arr[int(li)])
                if ci == qi or corpus_q_stripped[ci] == corpus_q_stripped[qi]: continue
                r1 = uni_r1(gold, uni_toks(answers_raw[ci])[:CAP])
                (pos if r1 >= 0.5 else neg if r1 <= 0.2 else []).append(ci)
            if pos and neg:
                pairs.append((questions_raw[qi], questions_raw[random.choice(pos)], 1))
                for n_ in random.sample(neg, min(2, len(neg))):
                    pairs.append((questions_raw[qi], questions_raw[n_], 0))
                made += 1
    print(f"Pairs: {len(pairs)}  (pos {sum(1 for p in pairs if p[2]==1)})")

    # ---- 2) TRAIN XLM-R-base cross-encoder (binary) ----
    from transformers import AutoTokenizer, AutoModelForSequenceClassification
    gc.collect(); torch.cuda.empty_cache()
    CE = 'xlm-roberta-base'
    ctok = AutoTokenizer.from_pretrained(CE)
    cmod = AutoModelForSequenceClassification.from_pretrained(CE, num_labels=2).to('cuda:0')
    opt = torch.optim.AdamW(cmod.parameters(), lr=2e-5)
    random.shuffle(pairs)
    B = 16
    cmod.train()
    for ep in range(1):
        pbar = tqdm(range(0, len(pairs), B), desc=f"CE epoch {ep+1}")
        for s in pbar:
            chunk = pairs[s:s+B]
            enc = ctok([p[0] for p in chunk], [p[1] for p in chunk], padding=True,
                       truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
            labels = torch.tensor([p[2] for p in chunk]).to('cuda:0')
            out = cmod(**enc, labels=labels)
            out.loss.backward(); opt.step(); opt.zero_grad()
            if s % (B*50) == 0: pbar.set_postfix(loss=float(out.loss))
    cmod.eval()
    cmod.save_pretrained(str(OUTPUT_DIR/'ce-reranker-v2')); ctok.save_pretrained(str(OUTPUT_DIR/'ce-reranker-v2'))

    # ---- 3) GUARDED EVAL: rerank top-15, per-language margin gate, split-half ----
    @torch.no_grad()
    def ce_scores(query, cand_questions):
        enc = ctok([query]*len(cand_questions), cand_questions, padding=True,
                   truncation=True, max_length=160, return_tensors='pt').to('cuda:0')
        return torch.softmax(cmod(**enc).logits, -1)[:, 1].cpu().numpy()

    print(f"\n{'Sub':<10} {'margin':>7} {'cur wR':>8} {'CE wR':>8} {'holdΔ':>8}  use")
    ce_gate = {}
    for sub in sorted(SUBSET_TO_LANG):
        tag, a, m = choice[sub]
        idxs = [i for i in range(len(val_df)) if str(val_df.iloc[i]['subset'])==sub
                and val_cands_all[i] and str(val_df.iloc[i]['output']).strip()]
        tune, hold = idxs[0::2][:250], idxs[1::2][:250]
        def eval_ce(ix, margin):
            tot = []
            for i in ix:
                pool = val_cands_all[i]
                cq = [questions_raw[c['idx']] for c in pool]
                cs = ce_scores(val_qs[i], cq)
                b = int(np.argmax(cs))
                pick = pool[b]['answer'] if (b != 0 and cs[b]-cs[0] > margin) else pool[0]['answer']
                # CE replaces TOP-1 only; downstream comb_mbr/stitch unchanged for now
                rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]; at = uni_toks(pick)[:CAP]
                tot.append(0.37*uni_r1(rt, at)+0.37*uni_rl(rt, at))
            return np.mean(tot)
        def eval_base(ix):
            tot = []
            for i in ix:
                rt = uni_toks(str(val_df.iloc[i]['output']))[:CAP]
                at = uni_toks(val_cands_all[i][0]['answer'])[:CAP]
                tot.append(0.37*uni_r1(rt, at)+0.37*uni_rl(rt, at))
            return np.mean(tot)
        best_m = max([0.1, 0.2, 0.3, 0.5], key=lambda mm: eval_ce(tune, mm))
        h_ce, h_b = eval_ce(hold, best_m), eval_base(hold)
        use = h_ce > h_b + 0.003
        ce_gate[sub] = (bool(use), best_m)
        print(f"{sub:<10} {best_m:>7.1f} {h_b:>8.4f} {h_ce:>8.4f} {h_ce-h_b:>+8.4f}  {'CE' if use else 'keep'}")
    print("Gate:", ce_gate)
else:
    print('CE reranker already loaded — skipping training')


## 5. Build V4 Base Submission

V4 uses the per-language strategy table:
- **Strong languages** (Eng_Uga, Eng_Ken, Eng_Eth): AfriE5 + FT2 interpolation → MBR top-1
- **Swa_Ken:** AfriE5 + FT2 interpolation (β=0.8) → MBR
- **Lug_Uga:** AfriE5 + per-language adapter (β=0.8) → MBR
- **Aka_Gha:** Extractive stitch from AfriE5 pool
- **Eng_Gha:** Qwen2.5-7B LoRA generation (loaded separately)

If V4 CSV already exists on Drive, this cell loads it directly.


In [ ]:
import pandas as pd, numpy as np
from tqdm import tqdm

V4_PATH = OUTPUT_DIR / 'submission_v4_final.csv'
if V4_PATH.exists():
    v4 = pd.read_csv(V4_PATH)
    print(f"V4 loaded from cache: {len(v4)} rows, score 0.6898")
else:
    raise FileNotFoundError(
        f"V4 submission not found at {V4_PATH}. "
        "Run the full training pipeline first (notebook 04 + 05 in the repo)."
    )

v4_map = dict(zip(v4['ID'].astype(str), v4['TargetR1F1']))
print(f"V4 base: {len(v4_map)} answers loaded")


## 6. Build V6 = V4 + CE-stitch (Aka_Gha, Amh_Eth) + Interpolated Retrieval

V6 upgrades V4 by:
1. **CE-reranked extractive stitch** for Aka_Gha and Amh_Eth (the two weakest languages)
2. **Embedding interpolation** for Lug_Uga (β=0.8: 80% AfriE5, 20% per-lang adapter)
3. **Embedding interpolation** for Swa_Ken (β=0.8: 80% AfriE5, 20% FT2)
4. Everything else unchanged from V4


In [ ]:
# =============================================================================
# V6: V4 + CE-stitch (Aka_Gha, Amh_Eth) + Lug_Uga adapter β=0.8 + Swa_Ken β=0.8
# Identical columns, open-source. Requires: bootstrap + cmod/ctok loaded.
# =============================================================================
import json, numpy as np, pandas as pd, torch
from tqdm import tqdm

ft2_corpus = np.load(CACHE/'ft2_corpus.npy'); ft2_test = np.load(CACHE/'ft2_test.npy')
pl_lug = np.load(CACHE/'pl_Lug_Uga_test.npy')
pl_lug_idx = json.load(open(CACHE/'pl_Lug_Uga_test_idx.json'))
pl_lug_map = {i: j for j, i in enumerate(pl_lug_idx)}
lug_corpus = np.load(CACHE/'pl_Lug_Uga_corpus.npy')   # saved during the adapter run
CE_LANGS = {'Aka_Gha', 'Amh_Eth'}

@torch.no_grad()
def ce_scores_t(query, cand_qs):
    enc = ctok([query]*len(cand_qs), cand_qs, padding=True, truncation=True,
               max_length=160, return_tensors='pt').to('cuda:0')
    return torch.softmax(cmod(**enc).logits, -1)[:, 1].cpu().numpy()

rows_v6 = []
v4 = pd.read_csv(OUTPUT_DIR/'submission_v4_final.csv')
v4_map = dict(zip(v4['ID'].astype(str), v4['TargetR1F1']))
for i in tqdm(range(len(test_df)), desc="V6"):
    rid = str(test_df.iloc[i]['ID']); sub = test_subs[i]
    if sub in CE_LANGS:
        pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                        k=K_CANDIDATES, exclude_exact=False)
        if pool:
            cs = ce_scores_t(test_qs[i], [questions_raw[c['idx']] for c in pool])
            order = np.argsort(-cs)
            cp = [{'answer': pool[j]['answer'], 'sim': float(cs[j]), 'idx': pool[j]['idx']}
                  for j in order]
            lam = uni_stitch_gate[sub]['lam'] if sub in uni_stitch_gate else 0.70
            ans = uni_stitch(cp, lam, sub)
        else:
            ans = v4_map[rid]
    elif sub == 'Lug_Uga':
        _, mask = lang_indices[sub]; mask_arr = np.array(mask)
        s = 0.8*(corpus_emb[mask_arr] @ test_emb[i]) + 0.2*(lug_corpus @ pl_lug[pl_lug_map[i]])
        order = np.argsort(-s)
        pool = [{'answer': answers_raw[int(mask_arr[j])], 'sim': float(s[j]),
                 'idx': int(mask_arr[j])} for j in order[:K_CANDIDATES]]
        tag, a, m = choice[sub]
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
    elif sub == 'Swa_Ken':
        _, mask = lang_indices[sub]; mask_arr = np.array(mask)
        s = 0.8*(corpus_emb[mask_arr] @ test_emb[i]) + 0.2*(ft2_corpus[mask_arr] @ ft2_test[i])
        order = np.argsort(-s)
        pool = [{'answer': answers_raw[int(mask_arr[j])], 'sim': float(s[j]),
                 'idx': int(mask_arr[j])} for j in order[:K_CANDIDATES]]
        tag, a, m = choice[sub]
        dd, ddw, u1, uL = uni_prep(pool)
        ans = dd[mbr_idx(0.5*u1+0.5*uL, ddw, a, m)]
    else:
        ans = v4_map[rid]    # everything else: byte-identical to V4
    rows_v6.append({'ID': test_df.iloc[i]['ID'], 'TargetR1F1': ans,
                    'TargetRLF1': ans, 'TargetLLM': ans})

df6 = pd.DataFrame(rows_v6).reindex(columns=SUB_COLS)
assert len(df6) == len(test_df) and not df6.isnull().any().any()
df6.to_csv(OUTPUT_DIR/'submission_v6.csv', index=False)
print("Saved: submission_v6.csv")

## 7. Build V7 = V6 + Amh_Eth QA-union CE-stitch ⭐

**This is the final submission (0.6908).**

V7 adds one more technique for Amharic:
- Concatenate question+answer text into a single embedding for each Amharic training answer
- Union this QA-embedding pool with the standard Q-only pool
- CE-rerank the combined pool
- Apply extractive stitch

Only ~60 Amharic rows change, but it pushes the score from 0.6898 → 0.6908.


In [ ]:
# =============================================================================
# V7 = V6 + Amh_Eth QA-union CE-stitch (only ~60 rows change)
# Requires: enc_model (AfriE5), cmod/ctok (ce-reranker-v2), bootstrap state.
# =============================================================================
import numpy as np, pandas as pd, torch
from tqdm import tqdm

sub = 'Amh_Eth'
_, mask = lang_indices[sub]; mask_arr = np.array(mask)
qa_texts = [f"passage: {questions_raw[ci]} {answers_raw[ci]}" for ci in mask_arr]
qa_emb = enc_model.encode(qa_texts, batch_size=32, normalize_embeddings=True,
                          show_progress_bar=True).astype(np.float32)
np.save(CACHE/'qa_emb_amh.npy', qa_emb)   # for reproducibility package

g_lam = uni_stitch_gate.get(sub, {'lam': 0.70})['lam']
t_idx = [i for i in range(len(test_df)) if test_subs[i] == sub]
new_ans = {}
for i in tqdm(t_idx, desc="Amh V7"):
    pool = get_same_lang_candidates(test_qs[i].strip(), test_emb[i], sub,
                                    k=K_CANDIDATES, exclude_exact=False)
    s = qa_emb @ test_emb[i]
    qa_pool = [{'answer': answers_raw[int(mask_arr[j])], 'sim': float(s[j]),
                'idx': int(mask_arr[j])} for j in np.argsort(-s)[:15]]
    seen = {c['idx'] for c in pool}
    upool = pool + [c for c in qa_pool if c['idx'] not in seen]
    if not upool: continue
    cs = ce_scores(test_qs[i], [questions_raw[c['idx']] for c in upool])
    cp = [{'answer': upool[j]['answer'], 'sim': float(cs[j]), 'idx': upool[j]['idx']}
          for j in np.argsort(-cs)]
    new_ans[str(test_df.iloc[i]['ID'])] = uni_stitch(cp, g_lam, sub)

v6 = pd.read_csv(OUTPUT_DIR/'submission_v6.csv')
sel = v6['ID'].astype(str).isin(new_ans)
for col in ['TargetR1F1','TargetRLF1','TargetLLM']:
    v6.loc[sel, col] = v6.loc[sel, 'ID'].astype(str).map(new_ans)
assert len(v6) == len(test_df) and not v6.isnull().any().any()
print(f"Rows changed: {sel.sum()}")
v6.to_csv(OUTPUT_DIR/'submission_v7.csv', index=False)
print("Saved: submission_v7.csv")

## 8. Validate Final Submission


In [ ]:
import pandas as pd

sub = pd.read_csv(OUTPUT_DIR / 'submission_v7.csv')
sample = pd.read_csv(DATA_DIR / 'SampleSubmission.csv')

# Validation checks
assert list(sub.columns) == list(sample.columns), f"Columns mismatch: {sub.columns} vs {sample.columns}"
assert len(sub) == len(sample), f"Row count mismatch: {len(sub)} vs {len(sample)}"
assert sub['ID'].equals(sample['ID']), "ID mismatch"
assert not sub.isnull().any().any(), "Contains nulls"
assert not (sub == '').any().any(), "Contains empty strings"

# Verify identical columns
assert (sub['TargetR1F1'] == sub['TargetRLF1']).all(), "R1F1 != RLF1"
assert (sub['TargetR1F1'] == sub['TargetLLM']).all(), "R1F1 != LLM"

print(f"✅ submission_v7.csv VALID")
print(f"   Rows: {len(sub)}")
print(f"   Columns: {list(sub.columns)}")
print(f"   Identical answer columns: True")
print(f"   Nulls/empty: None")
print(f"")
print(f"   Score: 0.6908 (public LB)")
print(f"   Breakdown: R1=0.681, RL=0.610, LLM=0.820")

# Show sample per language
for subset in sorted(sub.merge(pd.read_csv(DATA_DIR/'Test.csv')[['ID','subset']], on='ID')['subset'].unique()):
    merged = sub.merge(pd.read_csv(DATA_DIR/'Test.csv')[['ID','subset']], on='ID')
    n = (merged['subset'] == subset).sum()
    sample_ans = merged.loc[merged['subset'] == subset, 'TargetR1F1'].iloc[0]
    print(f"   {subset}: {n} rows, sample: {sample_ans[:80]}...")


## 9. Copy Submission to Working Directory


In [ ]:
import shutil
shutil.copy(OUTPUT_DIR / 'submission_v7.csv', '/content/submission_v7.csv')
print("Copied to /content/submission_v7.csv — ready for download")
